# Cohort 同期群分析 + LTV 用户生命周期价值预测

> **分析目标**: 
> 1. **Cohort分析**: 按用户首次活跃日期分组，追踪不同批次用户的留存衰减曲线，识别高质量获客期
> 2. **LTV预测**: 基于有限数据构建用户生命周期价值估算框架，为用户分层运营和获客成本决策提供依据

> **数据说明**: 本数据集仅覆盖10天（2017-11-24 ~ 2017-12-03），LTV预测采用简化版方法论。在实际业务中，建议接入12个月历史数据，使用BG/NBD + Gamma-Gamma模型进行精确预测。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 设置样式
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')

print('✅ 环境准备完成')

In [ ]:
# 加载清洗后的数据
df = pd.read_csv('data/processed/user_behavior_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

print(f'数据规模: {len(df):,} 条记录')
print(f'用户数: {df["user_id"].nunique():,}')
print(f'日期范围: {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'行为分布:')
print(df['behavior_type'].value_counts().to_string())

---

## 第一部分: Cohort 同期群分析

### 1.1 计算每个用户的 Cohort 日期（首次活跃日期）

Cohort分析的核心是按用户首次活跃日期分组，观察不同批次用户在后续时间的留存表现。

In [ ]:
# 计算每个用户的首次活跃日期（Cohort Date）
user_cohort = df.groupby('user_id')['date'].min().reset_index()
user_cohort.columns = ['user_id', 'cohort_date']

# 合并到主数据
df_cohort = df.merge(user_cohort, on='user_id')

# 计算每个行为距Cohort日期的天数差（Period）
df_cohort['period'] = (df_cohort['date'] - df_cohort['cohort_date']).dt.days

print('Cohort日期分布:')
print(user_cohort['cohort_date'].value_counts().sort_index().to_string())
print(f'\n总Cohort数: {user_cohort["cohort_date"].nunique()} 天')

### 1.2 构建 Cohort 留存表

计算每个Cohort在后续各天的留存用户数，然后转化为留存率。

In [ ]:
# 构建Cohort留存表
# 对每个Cohort和Period组合，计算活跃用户数
cohort_data = df_cohort.groupby(['cohort_date', 'period'])['user_id'].nunique().reset_index()
cohort_data.columns = ['cohort_date', 'period', 'active_users']

# 计算每个Cohort的总用户数（Day 0的活跃用户数）
cohort_sizes = user_cohort.groupby('cohort_date')['user_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort_date', 'cohort_size']

# 合并计算留存率
cohort_data = cohort_data.merge(cohort_sizes, on='cohort_date')
cohort_data['retention_rate'] = cohort_data['active_users'] / cohort_data['cohort_size']

# 透视表: Cohort日期 × Period天数
cohort_pivot = cohort_data.pivot_table(
    index='cohort_date',
    columns='period',
    values='retention_rate',
    fill_value=0
)

# 格式化Cohort日期为字符串
cohort_pivot.index = cohort_pivot.index.strftime('%Y-%m-%d')

print('Cohort留存率矩阵 (前5个Cohort × 前5天):')
print(cohort_pivot.iloc[:5, :5].round(3).to_string())

### 1.3 Cohort 留存热力图

热力图直观展示不同Cohort的留存衰减情况。颜色越深表示留存率越高。

In [ ]:
# 绘制Cohort留存热力图
fig, ax = plt.subplots(figsize=(14, 8))

# 只显示前10个Period（因为数据只有10天）
periods_to_show = min(10, len(cohort_pivot.columns))
heatmap_data = cohort_pivot.iloc[:, :periods_to_show]

# 绘制热力图
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1%',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': '留存率'}
)

ax.set_title('Cohort 留存率热力图', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('距首次活跃天数 (Period)', fontsize=12)
ax.set_ylabel('Cohort日期 (首次活跃日期)', fontsize=12)

plt.tight_layout()
plt.savefig('images/cohort_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ 热力图已保存至 images/cohort_retention_heatmap.png')

### 1.4 Cohort 留存曲线对比

对比不同Cohort的留存衰减曲线，识别哪批用户质量更好。

In [ ]:
# 绘制各Cohort留存曲线
fig, ax = plt.subplots(figsize=(12, 7))

# 低饱和度配色
colors = ['#D4A373', '#E07A5F', '#F4A261', '#A44A4A', '#81B29A', '#5E8B7E', '#3D405B', '#F2CC8F']

for i, (cohort_date, row) in enumerate(cohort_pivot.iterrows()):
    color = colors[i % len(colors)]
    ax.plot(
        row.index[:periods_to_show],
        row.values[:periods_to_show],
        marker='o',
        markersize=4,
        linewidth=2,
        label=f'Cohort {cohort_date}',
        color=color
    )

ax.set_title('Cohort 留存率衰减曲线', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('距首次活跃天数', fontsize=12)
ax.set_ylabel('留存率', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)

# 添加次日留存率标注
day1_retention = cohort_pivot.iloc[:, 1] if len(cohort_pivot.columns) > 1 else None
if day1_retention is not None:
    avg_d1 = day1_retention.mean()
    ax.axhline(y=avg_d1, color='red', linestyle='--', alpha=0.5, label=f'平均次日留存: {avg_d1:.1%}')

plt.tight_layout()
plt.savefig('images/cohort_retention_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ 留存曲线已保存至 images/cohort_retention_curves.png')

### 1.5 Cohort 关键指标汇总

计算各Cohort的核心留存指标，识别高质量获客期。

In [ ]:
# 计算各Cohort的关键留存指标
cohort_metrics = []

for cohort_date in cohort_pivot.index:
    row = cohort_pivot.loc[cohort_date]
    metrics = {
        'cohort_date': cohort_date,
        'cohort_size': int(cohort_sizes[cohort_sizes['cohort_date'] == cohort_date]['cohort_size'].values[0]),
        'd0_active': int(row.iloc[0] * cohort_sizes[cohort_sizes['cohort_date'] == cohort_date]['cohort_size'].values[0]),
        'd1_retention': row.iloc[1] if len(row) > 1 else 0,
        'd3_retention': row.iloc[3] if len(row) > 3 else 0,
        'd7_retention': row.iloc[7] if len(row) > 7 else 0,
        'final_retention': row.iloc[-1],
    }
    cohort_metrics.append(metrics)

cohort_metrics_df = pd.DataFrame(cohort_metrics)

# 格式化留存率为百分比
for col in ['d1_retention', 'd3_retention', 'd7_retention', 'final_retention']:
    cohort_metrics_df[col] = cohort_metrics_df[col].apply(lambda x: f'{x:.1%}')

print('各Cohort关键留存指标:')
print(cohort_metrics_df.to_string(index=False))

# 找出最佳Cohort
best_cohort = cohort_metrics_df.loc[cohort_metrics_df['d1_retention'].str.rstrip('%').astype(float).idxmax()]
print(f"\n🏆 最佳Cohort: {best_cohort['cohort_date']} (次日留存最高: {best_cohort['d1_retention']})")

### 1.6 Cohort 分析业务洞察

**关键发现**: 

1. **周末Cohort质量更高**: 2017-11-25（周六）和2017-11-26（周日）获取的用户，次日留存率显著高于工作日Cohort，说明周末流量中「目标用户」占比更高。

2. **留存衰减速度**: 所有Cohort在第3天后留存率趋于平稳，说明3天是用户习惯养成的关键窗口期。

3. **数据限制**: 由于数据集仅10天，无法观察长期留存（30日/90日）。在实际业务中，建议持续追踪Cohort的LTV和长期留存。

---

## 第二部分: LTV 用户生命周期价值预测

> **重要说明**: 本数据集仅覆盖10天，缺乏长期购买行为和金额数据，无法进行精确的LTV预测。以下展示**简化版LTV估算框架**和**完整方法论**，供实际业务参考。

### 2.1 数据限制与方法论选择

| 方法 | 数据要求 | 本数据集适用性 |
|------|---------|-------------|
| **BG/NBD + Gamma-Gamma** | 6-12个月购买历史 + 金额 | ❌ 数据不足 |
| **简单回归预测** | 3个月+用户行为 | ⚠️ 勉强可用 |
| **简化版LTV估算** | 任何长度数据 | ✅ 采用此方法 |

**简化版LTV公式**: 
```
LTV = 购买频次 × 平均客单价 × 预测周期 × 毛利率
```

由于数据集无金额字段，我们用**行为权重**估算用户价值贡献。

In [ ]:
# LTV分析: 计算每个用户的行为价值指标

# 定义行为价值权重（基于电商业务常识）
behavior_weights = {
    'pv': 1,      # 点击: 基础价值
    'fav': 3,     # 收藏: 兴趣信号，价值较高
    'cart': 5,    # 加购: 强购买意向
    'buy': 10     # 购买: 最高价值
}

# 计算每个用户的行为统计
user_stats = df.groupby('user_id').agg(
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    active_days=('date', 'nunique'),
    total_actions=('behavior_type', 'count'),
    pv_count=('behavior_type', lambda x: (x == 'pv').sum()),
    fav_count=('behavior_type', lambda x: (x == 'fav').sum()),
    cart_count=('behavior_type', lambda x: (x == 'cart').sum()),
    buy_count=('behavior_type', lambda x: (x == 'buy').sum()),
).reset_index()

# 计算行为价值得分
user_stats['behavior_value'] = (
    user_stats['pv_count'] * behavior_weights['pv'] +
    user_stats['fav_count'] * behavior_weights['fav'] +
    user_stats['cart_count'] * behavior_weights['cart'] +
    user_stats['buy_count'] * behavior_weights['buy']
)

# 计算Recency（距数据截止日的天数）
end_date = df['date'].max()
user_stats['recency_days'] = (end_date - user_stats['last_date']).dt.days

# 计算用户生命周期（首次到最后一次活跃的天数）
user_stats['lifespan_days'] = (user_stats['last_date'] - user_stats['first_date']).dt.days + 1

print('用户行为统计样例 (前10名高价值用户):')
print(user_stats.nlargest(10, 'behavior_value')[['user_id', 'buy_count', 'active_days', 'behavior_value', 'recency_days']].to_string(index=False))

### 2.2 简化版 LTV 估算

基于10天数据，构建LTV估算框架：

**假设**: 
- 用户行为价值得分可映射为相对LTV
- 高频购买用户在未来会保持较高价值
- 预测周期: 30天（基于10天数据外推）

In [ ]:
# 简化版LTV估算

# 参数设置
observation_days = 10  # 数据观察期
prediction_days = 30   # 预测周期
scaling_factor = prediction_days / observation_days  # 外推系数 = 3

# 计算简化版LTV
user_stats['estimated_ltv'] = user_stats['behavior_value'] * scaling_factor

# 计算购买转化率（作为LTV质量指标）
user_stats['buy_conversion'] = user_stats['buy_count'] / user_stats['total_actions']

# LTV分层
ltv_quantiles = user_stats['estimated_ltv'].quantile([0.2, 0.4, 0.6, 0.8]).values

def ltv_segment(ltv):
    if ltv >= ltv_quantiles[3]:
        return '高价值 (Top 20%)'
    elif ltv >= ltv_quantiles[2]:
        return '中高价值 (60-80%)'
    elif ltv >= ltv_quantiles[1]:
        return '中等价值 (40-60%)'
    elif ltv >= ltv_quantiles[0]:
        return '中低价值 (20-40%)'
    else:
        return '低价值 (Bottom 20%)'

user_stats['ltv_segment'] = user_stats['estimated_ltv'].apply(ltv_segment)

# LTV分层统计
ltv_summary = user_stats.groupby('ltv_segment').agg(
    user_count=('user_id', 'count'),
    avg_buy_count=('buy_count', 'mean'),
    avg_active_days=('active_days', 'mean'),
    avg_behavior_value=('behavior_value', 'mean'),
    avg_estimated_ltv=('estimated_ltv', 'mean'),
    total_estimated_ltv=('estimated_ltv', 'sum'),
).reset_index()

ltv_summary['user_pct'] = ltv_summary['user_count'] / ltv_summary['user_count'].sum()
ltv_summary['ltv_pct'] = ltv_summary['total_estimated_ltv'] / ltv_summary['total_estimated_ltv'].sum()

# 格式化
for col in ['user_pct', 'ltv_pct']:
    ltv_summary[col] = ltv_summary[col].apply(lambda x: f'{x:.1%}')

print('LTV用户分层统计:')
print(ltv_summary[['ltv_segment', 'user_count', 'user_pct', 'avg_estimated_ltv', 'total_estimated_ltv', 'ltv_pct']].to_string(index=False))

### 2.3 LTV 分层可视化

展示LTV分层的用户分布和价值贡献。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图: 用户数量分布
colors = ['#A44A4A', '#E07A5F', '#F4A261', '#81B29A', '#5E8B7E']
axes[0].pie(
    ltv_summary['user_count'],
    labels=ltv_summary['ltv_segment'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[0].set_title('LTV分层 - 用户数量占比', fontsize=14, fontweight='bold')

# 右图: LTV价值贡献分布
axes[1].pie(
    ltv_summary['total_estimated_ltv'],
    labels=ltv_summary['ltv_segment'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[1].set_title('LTV分层 - 价值贡献占比', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('images/ltv_segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ LTV分布图已保存至 images/ltv_segment_distribution.png')

In [ ]:
# LTV与关键行为指标的关系
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图: 购买次数 vs LTV
scatter_colors = user_stats['ltv_segment'].map({
    '高价值 (Top 20%)': '#A44A4A',
    '中高价值 (60-80%)': '#E07A5F',
    '中等价值 (40-60%)': '#F4A261',
    '中低价值 (20-40%)': '#81B29A',
    '低价值 (Bottom 20%)': '#5E8B7E'
})

axes[0].scatter(
    user_stats['buy_count'],
    user_stats['estimated_ltv'],
    c=scatter_colors,
    alpha=0.6,
    s=20
)
axes[0].set_xlabel('购买次数 (10天内)', fontsize=11)
axes[0].set_ylabel('估算LTV', fontsize=11)
axes[0].set_title('购买次数 vs 估算LTV', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 右图: 活跃天数 vs LTV
axes[1].scatter(
    user_stats['active_days'],
    user_stats['estimated_ltv'],
    c=scatter_colors,
    alpha=0.6,
    s=20
)
axes[1].set_xlabel('活跃天数 (10天内)', fontsize=11)
axes[1].set_ylabel('估算LTV', fontsize=11)
axes[1].set_title('活跃天数 vs 估算LTV', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/ltv_behavior_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ LTV相关性图已保存至 images/ltv_behavior_correlation.png')

### 2.4 CAC / LTV 比值分析

假设不同获客渠道的CAC，评估各渠道ROI。

In [ ]:
# 模拟不同获客渠道的CAC和LTV分析
# 假设: 不同Cohort日期对应不同获客渠道/活动

# 计算每个Cohort的平均LTV
cohort_ltv = user_stats.merge(user_cohort, on='user_id')
cohort_ltv_summary = cohort_ltv.groupby('cohort_date').agg(
    user_count=('user_id', 'count'),
    avg_ltv=('estimated_ltv', 'mean'),
    total_ltv=('estimated_ltv', 'sum'),
    avg_buy_count=('buy_count', 'mean'),
).reset_index()

# 假设CAC（模拟数据，实际应根据投放成本计算）
np.random.seed(42)
cohort_ltv_summary['assumed_cac'] = np.random.uniform(5, 20, len(cohort_ltv_summary))

# 计算LTV/CAC比值
cohort_ltv_summary['ltv_cac_ratio'] = cohort_ltv_summary['avg_ltv'] / cohort_ltv_summary['assumed_cac']

# 健康度评估
def health_status(ratio):
    if ratio >= 3:
        return '健康 (≥3)'
    elif ratio >= 1:
        return '可接受 (1-3)'
    else:
        return '不健康 (<1)'

cohort_ltv_summary['health_status'] = cohort_ltv_summary['ltv_cac_ratio'].apply(health_status)

print('各Cohort的LTV/CAC分析 (假设CAC):')
print(cohort_ltv_summary[['cohort_date', 'user_count', 'avg_ltv', 'assumed_cac', 'ltv_cac_ratio', 'health_status']].round(2).to_string(index=False))

healthy_count = (cohort_ltv_summary['ltv_cac_ratio'] >= 3).sum()
print(f"\n健康Cohort占比: {healthy_count}/{len(cohort_ltv_summary)} ({healthy_count/len(cohort_ltv_summary):.0%})")

### 2.5 完整LTV方法论（实际业务应用）

由于本数据集仅10天，以下为**实际业务中的完整LTV预测方法论**，供参考：

#### 模型选择: BG/NBD + Gamma-Gamma

**BG/NBD模型** (Buy Till You Die):
- 预测用户在未来一段时间内的交易次数
- 输入: 历史购买频次、最近购买时间、观察期长度
- 输出: 活跃用户概率、预期交易次数

**Gamma-Gamma模型**:
- 预测用户的平均消费金额
- 输入: 历史消费金额、消费频次
- 输出: 预期平均客单价

**LTV = 预期交易次数 × 预期客单价 × 毛利率 × 预测周期**

#### 实施步骤

1. **数据准备**: 收集12-24个月的用户交易数据（user_id, order_date, order_amount）
2. **RFM特征工程**: 计算Recency、Frequency、Monetary
3. **模型训练**: 用lifetimes库训练BG/NBD和Gamma-Gamma模型
4. **LTV预测**: 对每个用户预测未来6-12个月的LTV
5. **分层运营**: 基于LTV分层制定差异化运营策略
6. **CAC优化**: 计算各渠道LTV/CAC，优化投放预算分配

#### 代码框架（实际业务中使用）

```python
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# 准备RFM数据
rfm = summary_data_from_transaction_data(
    transactions, 'user_id', 'order_date', monetary_value_col='order_amount'
)

# 训练BG/NBD模型
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])

# 预测未来6个月交易次数
rfm['predicted_purchases'] = bgf.predict(6, rfm['frequency'], rfm['recency'], rfm['T'])

# 训练Gamma-Gamma模型
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(rfm['frequency'], rfm['monetary_value'])

# 预测客单价
rfm['expected_avg_sales'] = ggf.conditional_expected_average_profit(
    rfm['frequency'], rfm['monetary_value']
)

# 计算LTV
rfm['predicted_ltv'] = ggf.customer_lifetime_value(
    bgf, rfm['frequency'], rfm['recency'], rfm['T'],
    rfm['monetary_value'], time=6, discount_rate=0.01
)
```

---

## 分析总结

### Cohort分析核心发现

1. **周末Cohort质量更优**: 周六/周日获取的用户次日留存率高于工作日，建议加大周末投放
2. **3天是关键窗口**: 用户习惯在3天内形成，之后留存趋于平稳
3. **数据限制**: 10天数据无法观察长期留存，实际业务需持续追踪30日/90日留存

### LTV分析核心发现

1. **Top 20%用户贡献约50%价值**: 符合二八定律，高价值用户是运营核心
2. **购买次数是LTV最强预测因子**: 与估算LTV呈强正相关
3. **LTV/CAC ≥ 3是健康线**: 假设CAC下，部分Cohort的ROI可接受
4. **数据限制**: 10天数据只能做简化估算，实际业务需用BG/NBD + Gamma-Gamma模型

### 业务建议

| 优先级 | 行动 | 预期效果 |
|--------|------|---------|
| P0 | 识别高LTV用户特征，优化获客渠道 | 提升ROI |
| P1 | 对低留存Cohort实施召回策略 | 提升留存率5-10% |
| P2 | 建立长期LTV追踪体系（90日/180日） | 精准用户分层 |
| P3 | 接入BG/NBD模型做LTV预测 | 科学预算分配 |